1. Из ноутбуков по практике "Рекуррентные и одномерные сверточные нейронные сети" выберите лучшую сеть, либо создайте свою.
2. Запустите раздел "Подготовка"
3. Подготовьте датасет с параметрами `VOCAB_SIZE=20'000`, `WIN_SIZE=1000`, `WIN_HOP=100`, как в ноутбуке занятия, и обучите выбранную сеть. Параметры обучения можно взять из практического занятия. Для  всех обучаемых сетей в данной работе они должны быть одни и теже.
4. Поменяйте размер словаря tokenaizera (`VOCAB_SIZE`) на `5000`, `10000`, `40000`.  Пересоздайте датасеты, при этом оставьте `WIN_SIZE=1000`, `WIN_HOP=100`.
Обучите выбранную нейронку на этих датасетах.  Сделайте выводы об  изменении  точности распознавания авторов текстов. Результаты сведите в таблицу
5. Поменяйте длину отрезка текста и шаг окна разбиения текста на векторы  (`WIN_SIZE`, `WIN_HOP`) используя значения (`500`,`50`) и (`2000`,`200`). Пересоздайте датасеты, при этом оставьте `VOCAB_SIZE=20000`. Обучите выбранную нейронку на этих датасетах. Сделайте выводы об  изменении точности распознавания авторов текстов.

Результаты всей работы сведите в таблицу.

## Подготовка

In [ ]:
# Работа с массивами данных
import numpy as np

# Функции-утилиты для работы с категориальными данными
from tensorflow.keras import utils

# Класс для конструирования последовательной модели нейронной сети
from tensorflow.keras.models import Sequential

# Основные слои
from tensorflow.keras.layers import Dense, Dropout, SpatialDropout1D, BatchNormalization, Embedding, Flatten, Activation
from tensorflow.keras.layers import SimpleRNN, GRU, LSTM, Bidirectional, Conv1D, MaxPooling1D, GlobalMaxPooling1D

# Токенизатор для преобразование текстов в последовательности
from tensorflow.keras.preprocessing.text import Tokenizer

# Рисование схемы модели
from tensorflow.keras.utils import plot_model

# Матрица ошибок классификатора
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Загрузка датасетов из облака google
import gdown

# Функции операционной системы
import os

# Работа со временем
import time

# Регулярные выражения
import re

# Отрисовка графиков
import matplotlib.pyplot as plt

# Вывод объектов в ячейке colab
from IPython.display import display

%matplotlib inline

from sklearn.model_selection import train_test_split

In [ ]:
# Загрузим датасет из облака
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip', None, quiet=True)

'writers.zip'

In [ ]:
# Распакуем архив в папку writers
!unzip -o writers.zip -d writers/


Archive:  writers.zip
  inflating: writers/(Клиффорд_Саймак) Обучающая_5 вместе.txt  
  inflating: writers/(Клиффорд_Саймак) Тестовая_2 вместе.txt  
  inflating: writers/(Макс Фрай) Обучающая_5 вместе.txt  
  inflating: writers/(Макс Фрай) Тестовая_2 вместе.txt  
  inflating: writers/(О. Генри) Обучающая_50 вместе.txt  
  inflating: writers/(О. Генри) Тестовая_20 вместе.txt  
  inflating: writers/(Рэй Брэдберри) Обучающая_22 вместе.txt  
  inflating: writers/(Рэй Брэдберри) Тестовая_8 вместе.txt  
  inflating: writers/(Стругацкие) Обучающая_5 вместе.txt  
  inflating: writers/(Стругацкие) Тестовая_2 вместе.txt  
  inflating: writers/(Булгаков) Обучающая_5 вместе.txt  
  inflating: writers/(Булгаков) Тестовая_2 вместе.txt  


In [ ]:
FILE_DIR = 'writers'
SIG_TRAIN = 'обучающая'
SIG_TEST = 'тестовая'

def load_data(file_dir, sig_train, sig_test):
    class_list = []
    text_train = []
    text_test = []

    file_list = sorted(os.listdir(file_dir))
    for file_name in file_list:
        m = re.match(r'\((.+)\) (\S+)_', file_name)
        if not m: continue

        class_name = m[1]
        subset_name = m[2].lower()

        if class_name not in class_list:
            class_list.append(class_name)
            text_train.append("")
            text_test.append("")

        cls_idx = class_list.index(class_name)
        with open(f"{file_dir}/{file_name}", "r", encoding="utf-8") as f:
            text = f.read().replace("\n", " ")

        if sig_train in subset_name:
            text_train[cls_idx] += " " + text
        elif sig_test in subset_name:
            text_test[cls_idx] += " " + text

    return text_train, text_test, class_list

text_train, text_test, CLASS_LIST = load_data(FILE_DIR, SIG_TRAIN, SIG_TEST)
CLASS_COUNT = len(CLASS_LIST)

In [ ]:
def split_sequence(seq, win, hop):
    return [seq[i:i + win] for i in range(0, len(seq) - win + 1, hop)]

def vectorize(seq_list, win, hop):
    x, y = [], []
    for cls, seq in enumerate(seq_list):
        chunks = split_sequence(seq, win, hop)
        x.extend(chunks)
        y.extend([cls] * len(chunks))
    return np.array(x), np.array(y)

In [ ]:
# Контекстный менеджер для измерения времени операций
# Операция обертывается менеджером с помощью оператора with

class timex:
    def __enter__(self):
        # Фиксация времени старта процесса
        self.t = time.time()
        return self

    def __exit__(self, type, value, traceback):
        # Вывод времени работы
        print('Время обработки: {:.2f} с'.format(time.time() - self.t))

## Решение

In [ ]:
def build_model(vocab_size, class_count):

    model = Sequential([

        # Преобразование слов в плотные векторы
        Embedding(vocab_size, 64),

        SpatialDropout1D(0.3),

        # Двунаправленный GRU:
        # учитывает контекст слева и справа
        Bidirectional(
            GRU(
                128,
                return_sequences=True,
                kernel_initializer='orthogonal'
            )
        ),

        # Выделение наиболее важных признаков по всей последовательности
        GlobalMaxPooling1D(),


        BatchNormalization(),


        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.5),


        Dense(class_count, activation='softmax')
    ])

    # Оптимизатор RMSprop подходит для RNN/GRU задач
    opt = RMSprop(learning_rate=0.001, rho=0.9)

    # Компиляция модели
    model.compile(
        optimizer=opt,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
experiments = [
    ("baseline", 20000, 1000, 100),
    ("vocab_5k", 5000, 1000, 100),
    ("vocab_10k", 10000, 1000, 100),
    ("vocab_40k", 40000, 1000, 100),
    ("win_small", 20000, 500, 50),
    ("win_large", 20000, 2000, 200),
]


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
results = []

for name, v_size, w_size, hop in experiments:

    #пересоздаем токенизатор под каждый размер словаря
    tokenizer = Tokenizer(num_words=v_size, lower=True, oov_token="<UNK>")
    tokenizer.fit_on_texts(text_train)

    # Переводим тексты в последовательности индексов
    train_seq = tokenizer.texts_to_sequences(text_train)
    test_seq = tokenizer.texts_to_sequences(text_test)

    # Нарезаем окна
    x_train, y_train = vectorize(train_seq, w_size, hop)
    x_test, y_test = vectorize(test_seq, w_size, hop)

    # Создаем и обучаем модель
    model = build_model(v_size, CLASS_COUNT)

    early_stop = EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True)

    history = model.fit(
        x_train, y_train,
        validation_data=(x_test, y_test),
        epochs=10,
        batch_size=64,
        verbose=0,
        callbacks=[early_stop]
    )
    # Берем лучшую достигнутую точность
    val_acc = max(history.history['val_accuracy'])

    results.append({
        "Experiment": name,
        "Vocab": v_size,
        "Win": w_size,
        "Hop": hop,
        "Accuracy": round(val_acc, 4)
    })

In [ ]:
import pandas as pd
df_results = pd.DataFrame(results)
df_results

,Experiment,Vocab,Win,Hop,Accuracy
0,baseline,20000,1000,100,0.6792
1,vocab_5k,5000,1000,100,0.6026
2,vocab_10k,10000,1000,100,0.6632
3,vocab_40k,40000,1000,100,0.6284
4,win_small,20000,500,50,0.6553
5,win_large,20000,2000,200,0.6952


По результатам экспериментов можно сделать вывод, что наилучшую точность показала конфигурация WIN_SIZE = 2000 и WIN_HOP = 200. Accuracy в этом случае составила 0.6952, что оказалось выше базовой модели с параметрами VOCAB_SIZE = 20000, WIN_SIZE = 1000, WIN_HOP = 100.

Изменение размера словаря показало, что слишком маленький словарь ухудшает качество классификации. При VOCAB_SIZE = 5000 точность снизилась до 0.6026, поскольку модель теряет часть информации об особенностях словаря автора. При увеличении словаря до 10000 качество улучшилось до 0.6632, однако всё ещё осталось ниже базового результата. Увеличение словаря до 40000 также не дало прироста качества — accuracy составила 0.6284. Вероятно, большое количество редких слов добавляет шум и усложняет обучение модели.

Эксперименты с размером окна показали, что длина текстового фрагмента существенно влияет на качество распознавания авторов. Уменьшение окна до 500 снизило точность до 0.6553, так как модель получает меньше контекста и хуже определяет стиль автора. Увеличение окна до 2000 наоборот улучшило результат, потому что более длинный фрагмент текста содержит больше характерных особенностей авторского стиля.

Таким образом, для данной задачи наиболее эффективной оказалась конфигурация с большим размером окна (2000) и словарём 20000. Это показывает, что для распознавания авторов важнее наличие достаточного текстового контекста, чем чрезмерное увеличение словаря tokenizer.